# Autoregressive Word-Level Text Generation on H.P. Lovecraft's Fiction
**Foundations of Generative AI (ICT4403) - Assignment 2**
Apratim Sharma | Reg. No. 230911390

Four recurrent architectures - **SimpleRNN, Bidirectional SimpleRNN, LSTM, GRU** - are trained under identical conditions to predict the next word, and are then used to generate text from the same seeds at three sampling temperatures.

**Dataset:** *The Complete Works of H.P. Lovecraft* (public-domain stories, 1917-1935; compiled ebook hosted on the Internet Archive)
https://archive.org/details/TheCompleteWorksOfHPLovecraft_201412

**Notebook outline**
1. Setup and configuration (shared by all models)
2. Dataset: download, characteristics, preprocessing
3. Input-target sequence construction (autoregressive windows)
4. Models
5. Training (identical settings, timed)
6. Quantitative evaluation (loss, perplexity, accuracy, training time)
7. Text generation at three temperatures + qualitative/diversity metrics
8. Summary files for the README

> Run on a GPU runtime (Colab: Runtime -> Change runtime type -> T4 GPU). Estimated total runtime is 15-30 minutes on a T4 (an estimate, not a measurement).

## 1. Setup and configuration

In [ ]:
import os, re, time, json, math, random, unicodedata, urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)

# ---- Hyper-parameters: IDENTICAL for all four models ----
SEQ_LEN    = 20      # context window (words) used to predict the next word
VOCAB_SIZE = 6000    # most frequent training words kept; the rest -> <unk>
EMBED_DIM  = 128
UNITS      = 128     # recurrent units (per direction for the Bidirectional model)
DROPOUT    = 0.3
BATCH_SIZE = 256
LR         = 1e-3
EPOCHS     = int(os.environ.get("EPOCHS", 15))   # upper bound; early stopping decides
PATIENCE   = 2       # early stopping on validation loss
MAX_TOKENS = int(os.environ.get("MAX_TOKENS", 0))  # 0 = full corpus (use a small value only for quick tests)

# ---- Generation settings ----
TEMPERATURES = [0.5, 1.0, 1.5]
GEN_LEN      = 80    # words generated per sample
N_SAMPLES    = 5     # samples per (seed, temperature) used for the diversity metrics

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Dataset

**Source:** *The Complete Works of H.P. Lovecraft* (Internet Archive item `TheCompleteWorksOfHPLovecraft_201412`), compiled from public-domain texts of his original solo fiction (e.g. *The Call of Cthulhu*, *At the Mountains of Madness*, *The Dunwich Horror*, *The Shadow over Innsmouth*). The OCR plain-text version is downloaded below.

Why this corpus: a single author with a very distinctive style (long, adjective-heavy sentences; recurring vocabulary such as *eldritch*, *cyclopean*, *unnamable*) makes it easy to judge whether a model captures style, loses coherence, or falls into repetition.

In [ ]:
DATA_URLS = [
    "https://archive.org/download/TheCompleteWorksOfHPLovecraft_201412/The%20Complete%20Works%20of%20H.P.%20Lovecraft_djvu.txt",
    "https://archive.org/stream/TheCompleteWorksOfHPLovecraft_201412/The%20Complete%20Works%20of%20H.P.%20Lovecraft_djvu.txt",
]
RAW_PATH = "lovecraft_raw.txt"

if not os.path.exists(RAW_PATH):
    last_err = None
    for url in DATA_URLS:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as r:
                text = r.read().decode("utf-8", errors="ignore")
            if len(text) > 100_000:
                with open(RAW_PATH, "w", encoding="utf-8") as f:
                    f.write(text)
                print("Downloaded from", url)
                break
        except Exception as e:
            last_err = e
    else:
        raise RuntimeError(f"Could not download the dataset: {last_err}. "
                           f"Download the _djvu.txt file manually from the Internet Archive page and save it as {RAW_PATH}.")

with open(RAW_PATH, encoding="utf-8") as f:
    raw = f.read()
print(f"Raw file: {len(raw):,} characters")

# Trim the front matter (licence + table of contents): start at the 2nd occurrence of the first story title
hits = [m.start() for m in re.finditer(r"The Tomb", raw)]
start = hits[1] if len(hits) > 1 else (hits[0] if hits else 0)
raw = raw[start:]
print("--- first 400 characters after trimming ---")
print(raw[:400])

### 2.1 Cleaning and tokenisation

In [ ]:
CHAR_MAP = {0x2018: "'", 0x2019: "'", 0x201c: '"', 0x201d: '"', 0x2014: " ", 0x2013: " ", 0x2026: "."}

def clean_and_tokenize(text):
    text = text.translate(CHAR_MAP)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()   # Celephais, etc.
    text = text.replace("\r", "")
    text = re.sub(r"-\n\s*", "", text)                      # re-join words hyphenated across line breaks
    lines = [l for l in text.split("\n")
             if not re.fullmatch(r"\s*\d+\s*", l) and "http" not in l.lower()]   # page numbers / URLs
    text = " ".join(lines).lower()
    toks = re.findall(r"[a-z]+(?:'[a-z]+)?|[.,;:!?]", text)  # words (with apostrophes) + sentence punctuation as tokens
    out = []
    for t in toks:                                           # collapse runs of identical punctuation
        if out and t == out[-1] and t in ".,;:!?":
            continue
        out.append(t)
    return out

tokens = clean_and_tokenize(raw)
if MAX_TOKENS:
    tokens = tokens[:MAX_TOKENS]
print(f"{len(tokens):,} tokens")
print(" ".join(tokens[:60]))

### 2.2 Dataset characteristics

In [ ]:
PUNCT = set(".,;:!?")
word_tokens = [t for t in tokens if t not in PUNCT]
counts_all = Counter(tokens)
counts_words = Counter(word_tokens)

# average sentence length (words between . ! ?)
sent_lens, cur = [], 0
for t in tokens:
    if t in ".!?":
        if cur: sent_lens.append(cur)
        cur = 0
    elif t not in PUNCT:
        cur += 1

stats = {
    "Total tokens (words + punctuation)": len(tokens),
    "Word tokens": len(word_tokens),
    "Unique words": len(counts_words),
    "Type-token ratio": round(len(counts_words) / len(word_tokens), 4),
    "Words occurring once (hapax)": sum(1 for c in counts_words.values() if c == 1),
    "Mean sentence length (words)": round(float(np.mean(sent_lens)), 1),
    "Median sentence length (words)": float(np.median(sent_lens)),
}
stats_df = pd.DataFrame([(k, f"{v:,}" if isinstance(v, (int, np.integer)) else str(v)) for k, v in stats.items()],
                        columns=["Statistic", "Value"])
display(stats_df)
stats_df.to_csv(f"{OUT_DIR}/dataset_stats.csv", index=False)

top = counts_words.most_common(20)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar([w for w, _ in top], [c for _, c in top], color="#4c72b0")
ax[0].set_title("20 most frequent words"); ax[0].tick_params(axis="x", rotation=70)
freqs = np.array(sorted(counts_words.values(), reverse=True))
ax[1].loglog(np.arange(1, len(freqs) + 1), freqs, color="#c44e52")
ax[1].set_title("Rank-frequency (Zipf) plot"); ax[1].set_xlabel("rank"); ax[1].set_ylabel("frequency")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/dataset_characteristics.png", dpi=150); plt.show()

## 3. Input-target sequences for autoregressive modelling

The token stream is split **chronologically** into train (80%) / validation (10%) / test (10%) *before* windowing, so no window straddles two splits. The vocabulary is built from the training split only; every other word becomes `<unk>`.

For a window of `SEQ_LEN = 20` tokens, the input is `w[t-20 .. t-1]` and the target is `w[t]` (the next word). Sliding the window one token at a time gives one training example per position.

In [ ]:
n = len(tokens)
train_tok = tokens[: int(0.8 * n)]
val_tok   = tokens[int(0.8 * n): int(0.9 * n)]
test_tok  = tokens[int(0.9 * n):]

UNK = "<unk>"
train_counts = Counter(train_tok)
itos = [UNK] + [w for w, _ in train_counts.most_common(VOCAB_SIZE - 1)]
stoi = {w: i for i, w in enumerate(itos)}
V = len(itos)
UNK_ID = 0

encode = lambda toks: np.array([stoi.get(t, UNK_ID) for t in toks], dtype=np.int32)
train_ids, val_ids, test_ids = encode(train_tok), encode(val_tok), encode(test_tok)

print(f"Vocabulary size: {V:,}")
print(f"Training-token coverage by vocab : {100 * (1 - (train_ids == UNK_ID).mean()):.2f}%")
print(f"Test-token coverage by vocab     : {100 * (1 - (test_ids == UNK_ID).mean()):.2f}%")

def make_xy(ids):
    win = sliding_window_view(ids, SEQ_LEN + 1)
    return np.ascontiguousarray(win[:, :-1]), np.ascontiguousarray(win[:, -1])

X_train, y_train = make_xy(train_ids)
X_val,   y_val   = make_xy(val_ids)
X_test,  y_test  = make_xy(test_ids)
print(f"Sequences -> train {len(X_train):,} | val {len(X_val):,} | test {len(X_test):,}")

i = 1000
print("\nExample input :", " ".join(itos[j] for j in X_train[i]))
print("Example target:", itos[y_train[i]])

# Reference point: unigram (context-free) perplexity on the test targets
uni = np.bincount(train_ids, minlength=V).astype(np.float64) + 1.0
uni /= uni.sum()
unigram_ppl = float(np.exp(-np.mean(np.log(uni[y_test]))))
print(f"\nUnigram baseline test perplexity: {unigram_ppl:,.1f}")

## 4. Models

All four share the same skeleton: `Embedding(128) -> recurrent layer(128 units) -> Dropout(0.3) -> Dense(V, softmax)`; only the recurrent layer changes.

| Model | Recurrent layer | Note |
|---|---|---|
| SimpleRNN | `SimpleRNN(128)` | plain recurrence, prone to vanishing gradients |
| Bidirectional RNN | `Bidirectional(SimpleRNN(128))` | reads the *known* 20-word context forwards and backwards (no future words are used, so it is still a valid next-word predictor); output size 256 |
| LSTM | `LSTM(128)` | gated memory cell |
| GRU | `GRU(128)` | gated, fewer parameters than LSTM |

Because the Bidirectional layer outputs 2x128 features, its parameter count is higher; this is inherent to the architecture and is reported in the results.

In [ ]:
MODEL_NAMES = ["SimpleRNN", "BiRNN", "LSTM", "GRU"]

def build_model(kind):
    inp = keras.Input(shape=(SEQ_LEN,), dtype="int32")
    x = layers.Embedding(V, EMBED_DIM)(inp)
    if kind == "SimpleRNN":
        x = layers.SimpleRNN(UNITS)(x)
    elif kind == "BiRNN":
        x = layers.Bidirectional(layers.SimpleRNN(UNITS))(x)
    elif kind == "LSTM":
        x = layers.LSTM(UNITS)(x)
    elif kind == "GRU":
        x = layers.GRU(UNITS)(x)
    else:
        raise ValueError(kind)
    x = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(V, activation="softmax")(x)
    model = keras.Model(inp, out, name=kind)
    model.compile(
        optimizer=keras.optimizers.Adam(LR, clipnorm=1.0),     # clipping keeps the plain RNNs stable
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5")],
    )
    return model

for k in MODEL_NAMES:
    print(f"{k:10s} parameters: {build_model(k).count_params():,}")

## 5. Training (identical settings, timed)

In [ ]:
models, histories, train_info = {}, {}, {}

for kind in MODEL_NAMES:
    keras.utils.set_random_seed(SEED)           # same initial randomness for every run
    model = build_model(kind)
    stopper = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
    print(f"\n=== Training {kind} ===")
    t0 = time.perf_counter()
    hist = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                     epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=True,
                     callbacks=[stopper], verbose=2)
    elapsed = time.perf_counter() - t0
    n_ep = len(hist.history["loss"])
    models[kind], histories[kind] = model, hist.history
    train_info[kind] = {
        "params": model.count_params(),
        "epochs_run": n_ep,
        "best_epoch": int(np.argmin(hist.history["val_loss"])) + 1,
        "train_time_s": round(elapsed, 1),
        "sec_per_epoch": round(elapsed / n_ep, 1),
    }
    print(f"{kind}: {n_ep} epochs in {elapsed:.1f}s")

with open(f"{OUT_DIR}/histories.json", "w") as f:
    json.dump({k: {m: [float(v) for v in vals] for m, vals in h.items()} for k, h in histories.items()}, f)

## 6. Quantitative evaluation (held-out test split)

In [ ]:
rows = []
for kind in MODEL_NAMES:
    loss, acc, top5 = models[kind].evaluate(X_test, y_test, batch_size=1024, verbose=0)
    rows.append({
        "Model": kind,
        "Parameters": train_info[kind]["params"],
        "Test loss": round(loss, 4),
        "Test perplexity": round(math.exp(loss), 2),
        "Top-1 acc (%)": round(100 * acc, 2),
        "Top-5 acc (%)": round(100 * top5, 2),
        "Epochs run": train_info[kind]["epochs_run"],
        "Best epoch": train_info[kind]["best_epoch"],
        "Train time (s)": train_info[kind]["train_time_s"],
        "Sec / epoch": train_info[kind]["sec_per_epoch"],
    })
results = pd.DataFrame(rows)
results.to_csv(f"{OUT_DIR}/model_comparison.csv", index=False)
display(results)
print(f"Reference: unigram baseline perplexity = {unigram_ppl:,.1f}")

In [ ]:
colors = {"SimpleRNN": "#c44e52", "BiRNN": "#dd8452", "LSTM": "#4c72b0", "GRU": "#55a868"}
fig, ax = plt.subplots(2, 2, figsize=(13, 9))

for k in MODEL_NAMES:
    h = histories[k]; ep = np.arange(1, len(h["loss"]) + 1)
    ax[0, 0].plot(ep, h["loss"], color=colors[k], label=k)
    ax[0, 1].plot(ep, h["val_loss"], color=colors[k], label=k)
ax[0, 0].set_title("Training loss"); ax[0, 1].set_title("Validation loss")
for a in ax[0]: a.set_xlabel("epoch"); a.set_ylabel("cross-entropy"); a.legend()

ax[1, 0].bar(results["Model"], results["Test perplexity"], color=[colors[m] for m in results["Model"]])
ax[1, 0].set_title("Test perplexity (lower is better)")
ax[1, 1].bar(results["Model"], results["Train time (s)"], color=[colors[m] for m in results["Model"]])
ax[1, 1].set_title("Total training time (s)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/training_comparison.png", dpi=150); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(results["Model"], results["Top-1 acc (%)"], color=[colors[m] for m in results["Model"]]); ax[0].set_title("Test top-1 accuracy (%)")
ax[1].bar(results["Model"], results["Top-5 acc (%)"], color=[colors[m] for m in results["Model"]]); ax[1].set_title("Test top-5 accuracy (%)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/accuracy_comparison.png", dpi=150); plt.show()

## 7. Text generation and temperature analysis

Generation is autoregressive: predict a distribution over the next word, sample one word, append it to the window, repeat.

Sampling with temperature `T` rescales the logits: `p_i ∝ exp(log p_i / T)`. Low `T` (0.5) sharpens the distribution (safer, more repetitive); `T = 1.0` samples from the model's own distribution; high `T` (1.5) flattens it (more varied, less coherent). `<unk>` is never sampled.

**Fair comparison:** every model receives the *same three seed sequences* and the *same random numbers* (Gumbel-max sampling with a fixed generator per temperature).

Quantitative text metrics (computed on the generated words only, over `3 seeds x 5 samples` per model/temperature):
- **distinct-1 / distinct-2** - share of unique words / word pairs (higher = more diverse)
- **repeated-trigram rate** - share of 3-grams that already appeared earlier in the same sample (higher = more repetitive)
- **corpus 4-gram overlap** - share of generated 4-grams that appear verbatim in the training text (very high = memorising/copying)
- **seed-topic retention** - share of the seed's content words that reappear in the output (rough proxy for contextual consistency)
- **punctuation rate** - share of generated tokens that are punctuation (sanity check on sentence structure)

In [ ]:
SEEDS = [
    "the old house stood at the end of the lane and the villagers said that something was moving in the cellar beneath the stairs",
    "i do not know how long i had been walking through the dark corridor when i first heard the distant sound of chanting from below",
    "the professor had spent many years studying the strange carvings found upon the ancient stones and he now feared that the truth was",
]

def seed_to_ids(text):
    toks = clean_and_tokenize(text)
    assert len(toks) >= SEQ_LEN, "seed must contain at least SEQ_LEN tokens"
    toks = toks[-SEQ_LEN:]
    oov = [t for t in toks if t not in stoi]
    if oov: print("  note: seed words mapped to <unk>:", oov)
    return toks, encode(toks)

seed_tokens, seed_ids = [], []
for s in SEEDS:
    t, i_ = seed_to_ids(s); seed_tokens.append(t); seed_ids.append(i_)

def generate_batch(model, start_ids, temperature, n_words, rng):
    """start_ids: (B, SEQ_LEN) int array. Returns (B, n_words) generated ids."""
    window = start_ids.copy()
    out = np.zeros((len(window), n_words), dtype=np.int32)
    for step in range(n_words):
        probs = model(window, training=False).numpy().astype(np.float64)
        logp = np.log(probs + 1e-12)
        logp[:, UNK_ID] = -np.inf                       # never emit <unk>
        z = logp / temperature + rng.gumbel(size=logp.shape)   # Gumbel-max == sampling from softmax(logp / T)
        nxt = z.argmax(axis=1).astype(np.int32)
        out[:, step] = nxt
        window = np.concatenate([window[:, 1:], nxt[:, None]], axis=1)
    return out

def detok(tokens_):
    s = " ".join(tokens_)
    s = re.sub(r"\s+([.,;:!?])", r"\1", s)
    return s

# batch layout: [seed0 x N_SAMPLES, seed1 x N_SAMPLES, seed2 x N_SAMPLES]
batch_start = np.repeat(np.stack(seed_ids), N_SAMPLES, axis=0)
generated = {}     # (model, T) -> (3*N_SAMPLES, GEN_LEN) ids
for kind in MODEL_NAMES:
    for ti, T in enumerate(TEMPERATURES):
        rng = np.random.default_rng(SEED + ti)           # same random numbers for every model
        generated[(kind, T)] = generate_batch(models[kind], batch_start, T, GEN_LEN, rng)
print("generation done")

In [ ]:
STOP = set("the a an and of to in that it was he she his her i my me we they them with for on at by as from but or not had have has be were is this which there what when who then than so if all one into upon over".split())
train_4grams = set(zip(*[train_tok[i:] for i in range(4)]))

def ngrams(seq, n): return list(zip(*[seq[i:] for i in range(n)]))

def text_metrics(ids_row, seed_toks):
    toks = [itos[i] for i in ids_row]
    d1 = len(set(toks)) / len(toks)
    bi = ngrams(toks, 2); d2 = len(set(bi)) / len(bi)
    tri = ngrams(toks, 3)
    seen, rep = set(), 0
    for g in tri:
        if g in seen: rep += 1
        seen.add(g)
    rep_rate = rep / len(tri)
    four = ngrams(toks, 4)
    overlap = sum(g in train_4grams for g in four) / len(four)
    seed_content = {t for t in seed_toks if t not in STOP and t not in PUNCT and len(t) > 3}
    retention = len(seed_content & set(toks)) / max(1, len(seed_content))
    punct = sum(t in PUNCT for t in toks) / len(toks)
    return d1, d2, rep_rate, overlap, retention, punct

rows = []
for kind in MODEL_NAMES:
    for T in TEMPERATURES:
        g = generated[(kind, T)]
        ms = [text_metrics(g[b], seed_tokens[b // N_SAMPLES]) for b in range(len(g))]
        m = np.mean(ms, axis=0)
        rows.append({"Model": kind, "Temperature": T,
                     "distinct-1": round(m[0], 3), "distinct-2": round(m[1], 3),
                     "repeated-trigram rate": round(m[2], 3), "corpus 4-gram overlap": round(m[3], 3),
                     "seed-topic retention": round(m[4], 3), "punctuation rate": round(m[5], 3)})
gen_metrics = pd.DataFrame(rows)
gen_metrics.to_csv(f"{OUT_DIR}/generation_metrics.csv", index=False)
display(gen_metrics)

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for k in MODEL_NAMES:
    d = gen_metrics[gen_metrics.Model == k]
    ax[0].plot(d.Temperature, d["distinct-2"], "o-", color=colors[k], label=k)
    ax[1].plot(d.Temperature, d["repeated-trigram rate"], "o-", color=colors[k], label=k)
    ax[2].plot(d.Temperature, d["corpus 4-gram overlap"], "o-", color=colors[k], label=k)
for a, t in zip(ax, ["distinct-2 (diversity)", "repeated-trigram rate (repetition)", "corpus 4-gram overlap (copying)"]):
    a.set_title(t); a.set_xlabel("temperature"); a.set_xticks(TEMPERATURES); a.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/temperature_effects.png", dpi=150); plt.show()

### 7.1 Generated samples (same seeds, all models, all temperatures)

In [ ]:
md_lines = ["# Generated text samples", "",
            f"Each sample is {GEN_LEN} words. Same seeds and same random numbers for every model. Text is lower-case because the corpus was lower-cased.", ""]
for si, st in enumerate(seed_tokens):
    md_lines += [f"## Seed {si + 1}", "", f"> {detok(st)}", ""]
    for T in TEMPERATURES:
        md_lines += [f"### Seed {si + 1} - temperature {T}", ""]
        for kind in MODEL_NAMES:
            g = generated[(kind, T)][si * N_SAMPLES]          # first sample for this seed
            md_lines += [f"**{kind}:** {detok([itos[i] for i in g])}", ""]
with open(f"{OUT_DIR}/generated_samples.md", "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines))

# show seed 1 inline
for T in TEMPERATURES:
    print(f"\n######## temperature {T}  |  seed: {detok(seed_tokens[0])}")
    for kind in MODEL_NAMES:
        print(f"\n[{kind}] {detok([itos[i] for i in generated[(kind, T)][0]])}")

## 8. Summary files for the README

In [ ]:
def md_table(df):
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join(lines)

summary = ["## Dataset statistics", "", md_table(stats_df), "",
           f"Vocabulary size: {V:,} | train/val/test sequences: {len(X_train):,} / {len(X_val):,} / {len(X_test):,} | unigram-baseline perplexity: {unigram_ppl:,.1f}", "",
           "## Model comparison (test split)", "", md_table(results), "",
           "## Generated-text metrics by temperature", "", md_table(gen_metrics), ""]
with open(f"{OUT_DIR}/results_summary.md", "w", encoding="utf-8") as f:
    f.write("\n".join(summary))
print("\n".join(summary))
print("\nFiles written to ./outputs :", sorted(os.listdir(OUT_DIR)))